In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# geometrische parameters

r2  = 0.15
r1  = 0.15
r5l = 0.12
r5h = 0.12
r4l = 0.12
r4h = 0.12

phi2_init  = 0.0
phi4h_init =  math.pi/3
phi4l_init = -math.pi/3

r3l = math.sqrt((r1 + r2*math.cos(phi2_init) + r4l*math.cos(phi4l_init))**2 +
                (      r2*math.sin(phi2_init) + r4l*math.sin(phi4l_init))**2)

r3h = math.sqrt((r1 + r2*math.cos(phi2_init) + r4h*math.cos(phi4h_init))**2 +
                (      r2*math.sin(phi2_init) + r4h*math.sin(phi4h_init))**2)

r3h_ext = r3h

phi3h_init = math.atan2((r2*math.sin(phi2_init) + r4h*math.sin(phi4h_init)),
                        (r1 + r2*math.cos(phi2_init) + r4h*math.cos(phi4h_init)))

phi3l_init = math.atan2((r2*math.sin(phi2_init) + r4l*math.sin(phi4l_init)),
                        (r1 + r2*math.cos(phi2_init) + r4l*math.cos(phi4l_init)))

# Crank-slider
r_crank  = 0.12
L_bielle = 0.25
F = np.array([-0.36, -0.32])

theta_crank_init = 0


# ==========================================================
# TEMPS
# ==========================================================

t_begin = 0.0
t_end   = 25.0
Ts      = 0.05
t = np.arange(t_begin, t_end + Ts, Ts)

# ==========================================================
# MOUVEMENT IMPOSE DU PISTON P
# ==========================================================

A = 0.1  # amplitude 10 cm

P_y = A * np.sin(2*np.pi*0.5*t)   # fréquence 0.5 Hz
P_x = np.zeros_like(t)

# ==========================================================
# ROTATION HELPER
# ==========================================================

def rotate_vector(z, theta):
    R = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta),  np.cos(theta)]])
    return R @ z

# point E
C = np.array([0.0, 0.0])
E = C + rotate_vector(
    np.array([r3h_ext, 0.0]),
    phi3h_init + np.pi
)

# point Q
Q = F + rotate_vector(
    np.array([r_crank, 0.0]),
    theta_crank_init
)

# vecteur QE
QE = E - Q

# angle de la bielle
phi_bielle_init = np.arctan2(QE[1], QE[0])

# ==========================================================
# SYSTEME
# ==========================================================

def system_equations(x, P_y_k):
    phi2, phi3h, phi4h, phi3l, phi4l, phi_bielle, theta_crank = x

    # loop haute
    F1 = r1 + r2*np.cos(phi2) + r4h*np.cos(phi4h) - r3h*np.cos(phi3h)
    F2 =      r2*np.sin(phi2) + r4h*np.sin(phi4h) - r3h*np.sin(phi3h)

    # loop basse
    F3 = r1 + r2*np.cos(phi2) + r4l*np.cos(phi4l) - r3l*np.cos(phi3l)
    F4 =      r2*np.sin(phi2) + r4l*np.sin(phi4l) - r3l*np.sin(phi3l)

    F5 = r2*np.sin(phi2) + r4h*np.sin(phi4h) + r5h*np.sin(phi4l) - P_y_k

    # point géométrique E
    C = np.array([0.0, 0.0])
    E = C + rotate_vector(np.array([r3h_ext, 0.0]), phi3h + np.pi)

    # contrainte bielle QE (remplace crank)
    F6 = E[0] - L_bielle*np.cos(phi_bielle) - r_crank*np.cos(theta_crank) - F[0]
    F7 = E[1] - L_bielle*np.sin(phi_bielle) - r_crank*np.sin(theta_crank) - F[1]

    return [F1, F2, F3, F4, F5, F6, F7]

# ==========================================================
# INITIALISATION
# ==========================================================

N = len(t)

phi2  = np.zeros(N)
phi3h = np.zeros(N)
phi4h = np.zeros(N)
phi3l = np.zeros(N)
phi4l = np.zeros(N)
phi_bielle = np.zeros(N)
theta_crank = np.zeros(N)

dphi2  = np.zeros(N)
dphi3h = np.zeros(N)
dphi4h = np.zeros(N)
dphi3l = np.zeros(N)
dphi4l = np.zeros(N)

ddphi2  = np.zeros(N)
ddphi3h = np.zeros(N)
ddphi4h = np.zeros(N)
ddphi3l = np.zeros(N)
ddphi4l = np.zeros(N)

x_guess = np.array([phi2_init, phi3h_init, phi4h_init, phi3l_init, phi4l_init, phi_bielle_init, theta_crank_init])

# ==========================================================
# LOOP
# ==========================================================

for k in range(N):

    sol = fsolve(lambda x: system_equations(x, P_y[k]), x_guess)

    phi2[k], phi3h[k], phi4h[k], phi3l[k], phi4l[k], phi_bielle[k], theta_crank[k] = sol
    x_guess = sol

    phi2_k, phi3h_k, phi4h_k, phi3l_k, phi4l_k, phi_bielle_k, theta_crank_k = sol

# ==========================================================
# ANIMATIE
# ==========================================================

x_left   = -0.55
x_right  =  0.50
y_bottom = -0.50
y_top    =  0.30

C = np.array([0.0, 0.0])

plt.ioff()

fig, ax = plt.subplots()

t_size = len(t)

frames = t_size / 5

delta = int(np.floor(t_size / frames))

index_vec = np.arange(0, t_size, delta, dtype=int)


def update(frame_idx):

    k = index_vec[frame_idx]

    ax.clear()

    ax.set_aspect('equal', adjustable='box')   # Tegen WARNING meldingen

    ax.set_xlabel('[m]')
    ax.set_ylabel('[m]')

    ax.set_xlim(x_left, x_right)
    ax.set_ylim(y_bottom, y_top)

    ax.set_title(f'Frame {k}')

    # Base : C -> A -> B
    A = C + np.array([r1, 0.0])
    B = A + rotate_vector(np.array([r2, 0.0]), phi2[k])

    # High / low branch
    D_h = B + rotate_vector(np.array([r4h, 0.0]), phi4h[k])
    D_l = B + rotate_vector(np.array([r4l, 0.0]), phi4l[k])

    P = D_h + (D_l - B)

    loop_ground  = np.array([C, A, B])
    loop_rhombus = np.array([B, D_h, P, D_l, B])

    loop_r3h = np.array([C, D_h])
    loop_r3l = np.array([C, D_l])

    ax.plot(loop_ground[:,0], loop_ground[:,1], '-o')

    ax.plot(loop_rhombus[:,0], loop_rhombus[:,1], '-o')

    ax.plot(loop_r3h[:,0], loop_r3h[:,1],
            '-o', color='purple', linewidth=2)

    ax.plot(loop_r3l[:,0], loop_r3l[:,1],
            '-o', color='green', linewidth=2)

    # verticale stippellijn
    xP = P[0]

    ax.plot([xP, xP],
            [y_bottom, y_top],
            '--',
            color='lightgray',
            linewidth=1)

    # Extensie
    E = C + rotate_vector(np.array([r3h_ext, 0.0]),
                          phi3h[k] + np.pi)

    ax.plot([C[0], E[0]],
            [C[1], E[1]],
            '-o',
            color='purple',
            linewidth=2)

    # Crank-slider
    Q = F + rotate_vector(np.array([r_crank, 0.0]),
                          theta_crank[k])

    ax.plot([F[0], Q[0]],
            [F[1], Q[1]],
            '-o',
            color='green',
            linewidth=2)

    ax.plot([Q[0], E[0]],
            [Q[1], E[1]],
            '-o',
            color='green',
            linewidth=2)

    return []


ani = FuncAnimation(
    fig,
    update,
    frames=len(index_vec),
    interval=50,
    blit=False
)

plt.close(fig)   # vermijdt parasitaire frames

display(HTML(ani.to_jshtml()))
